# DocTune - train your model

This notebook fine-tunes the model for your DocTune project on Google Colab's **free T4 GPU**, in your own Google account.

**Before you start**
1. **Runtime -> Change runtime type -> T4 GPU -> Save.**
2. Run the cells **one by one, from top to bottom** (or Runtime -> Run all) and **keep this tab open** while it trains. Colab disconnects idle tabs.
3. Have your **DocTune job token** ready (shown on your project page when you clicked *Start training*).

**Good to know**
- Progress is saved about every 7 minutes. If Colab stops (session limit, no GPU, closed tab), **open this notebook again and run all cells** - training continues from the last checkpoint.
- Qwen3.5 models spend about **5-6 minutes compiling kernels** at the start of each session. That's normal.
- DocTune never sees your Google account. Your job token, and your Hugging Face token if you enter one, stay in this notebook and are not saved.
- One training job per notebook. You can stop any time with the stop button (progress is saved first).

In [ ]:
# 1. Install DocTune's training package (pins Unsloth, PyTorch, TRL and PEFT to tested versions). ~2-4 minutes.
%pip install -q "doctune-nb[train] @ git+https://github.com/doctune/doctune@nb-v0.1.0#subdirectory=notebook"
# If Colab asks you to restart the session after installing, click "Restart session", then continue with cell 2.

In [ ]:
# 2. Your DocTune job token (paste it in the box; it is not shown or saved)
from getpass import getpass

import doctune_nb
from doctune_nb import DocTuneClient

API_URL = "https://api.doctune.example"  # <-- RELEASE: set to the production DocTune API URL before tagging

client = DocTuneClient(getpass("Paste your DocTune job token: "), API_URL)
print(f"DocTune notebook {doctune_nb.__version__} - token accepted locally.")

In [ ]:
# 3. Check the GPU
from doctune_nb import env as dt_env

env = dt_env.check(client)  # no GPU -> tells DocTune and stops here

In [ ]:
# 4. Load your training job from DocTune
from doctune_nb import ui

cfg = client.config()
ui.show_config(cfg, client.checkpoint_latest())

### 5. Optional: Hugging Face token
Only needed if your base model is **gated** on Hugging Face (for example Llama), or if you want to upload exports to **your** Hugging Face account. Leave it blank and press Enter to skip. The token stays in this notebook; DocTune never receives it.

To upload to Hugging Face, create a token with *write* access at https://huggingface.co/settings/tokens.

In [ ]:
# 5. Optional Hugging Face token (blank = skip)
HF_TOKEN = getpass("Hugging Face token (optional, press Enter to skip): ").strip() or None
print("Hugging Face token entered." if HF_TOKEN else "No Hugging Face token - skipping Hugging Face features.")

### 6. Optional: Google Drive
Mount your Google Drive if you want the GGUF file (for Ollama / llama.cpp, about 2.8 GB for a 4B model) saved there at the end. Colab will ask you to sign in; DocTune never sees this. Set `USE_DRIVE = False` to skip.

In [ ]:
# 6. Optional: mount Google Drive for exports
USE_DRIVE = True

if USE_DRIVE and dt_env.detect_platform() == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
else:
    print("Google Drive not mounted - the GGUF export will be skipped (your model still goes to DocTune).")

In [ ]:
# 7. Train (shows a progress bar; checkpoints are saved automatically). Stop button = save + pause.
from doctune_nb import train

result = train.run(client, cfg, env, hf_token=HF_TOKEN)

In [ ]:
# 8. Evaluate: base model vs your fine-tuned model on held-out questions (a few minutes)
from doctune_nb import evaluate

eval_result = evaluate.run(client, cfg, result)

In [ ]:
# 9. Upload your fine-tuned adapter to DocTune -> your model is ready in DocTune
from doctune_nb import export

export.upload_adapter_and_finish(client, cfg, result, eval_result, hf_token=HF_TOKEN)

### 10. Optional: GGUF file for Ollama / llama.cpp, merged weights
Your model is already in DocTune after cell 9. This cell is optional and slow (**about 20-25 minutes** for the GGUF file - it builds llama.cpp first).

- `GGUF_TO = "drive"` saves a Q4_K_M GGUF to *My Drive/DocTune/<project>/gguf/* (needs cell 6), `"hf"` uploads it to a private repo in your Hugging Face account (needs cell 5), `None` skips it.
- `PUSH_MERGED_TO_HF = True` also uploads full merged 16-bit weights (about 9 GB for 4B) to your Hugging Face account. Not offered for Drive: it would fill most of a free 15 GB Drive.
- 8B models: exports are experimental on free Colab (memory); set `ALLOW_EXPERIMENTAL = True` to try.

In [ ]:
# 10. Optional exports (each one is checked for free space / memory first and skipped with a message if it won't fit)
GGUF_TO = "drive"  # "drive", "hf" or None
PUSH_MERGED_TO_HF = False
HF_REPO = ""  # optional, e.g. "your-name/my-model"; blank = <your HF user>/<model name>
ALLOW_EXPERIMENTAL = False

exports = export.export_optional(
    client,
    cfg,
    result,
    eval_result,
    gguf_to=GGUF_TO,
    push_merged_to_hf=PUSH_MERGED_TO_HF,
    hf_token=HF_TOKEN,
    hf_repo=HF_REPO or None,
    allow_experimental=ALLOW_EXPERIMENTAL,
)

## Done

Go back to your project page in DocTune (link printed by cell 9): you'll see the evaluation, can download the adapter, and try the model in the playground.

**Run it on your own computer (faster than the web playground):** install [Ollama](https://ollama.com), download the GGUF folder from your Drive (or Hugging Face), open a terminal in it and run the two commands printed by cell 10:

```bash
ollama create <model-name> -f Modelfile
ollama run <model-name>
```

The `Modelfile` already contains the right chat template and settings. You can now close this tab - Colab frees the GPU when you disconnect (Runtime -> Disconnect and delete runtime).